# EU Conventions Tree Traversal Examples

This notebook shows how to use the LATTICE LLM-guided traversal flow on the EU conventions semantic tree.

It is intentionally example-oriented:
- no gold paths
- no evaluation metrics
- no tests
- just load a tree, run example queries, inspect the traversal, and visualize the prediction tree

Before running the traversal cells, make sure the API key for your selected LLM backend is available in the environment.


In [ ]:
from __future__ import annotations

import asyncio
import os
import pickle
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    while current != current.parent:
        if (current / "src").exists() and (current / "trees").exists():
            return current
        current = current.parent
    raise RuntimeError("Could not find the repository root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from google.genai import types as genai_types

from hyperparams import HyperParams
from llm_apis import GenAIAPI, OpenAIResponsesAPI, VllmAPI, LocalModelAPI
from prompts import get_traversal_prompt_response_constraint
from tree_construction.build_llm_bottom_up_tree import run_coro_sync
from tree_objects import InferSample, SemanticNode
from utils import compute_node_registry, get_all_leaf_nodes_with_path, post_process, setup_logger, visualize_sample

REPO_ROOT


In [ ]:
# Configuration
#
# TREE_PATH prefers the expected EU conventions tree path. The fallback points to
# the tree path produced by the current bottom-up construction notebook in this repo.
default_tree_path = REPO_ROOT / "trees" / "EU" / "eu_conventions_notebook" / "eu_conventions_tree-bottom-up-llm.pkl"
if not default_tree_path.exists():
    default_tree_path = REPO_ROOT / "trees" / "EU" / "codigo_civil_notebook" / "eu_conventions_tree-bottom-up-llm.pkl"

# HyperParams.from_args(...) mirrors the way run.py configures retrieval.
# We pass the required args first, then override/add notebook-specific values.
hp = HyperParams.from_args("--subset fiqa --tree_version eu_conventions_notebook")
hp.TREE_PATH = str(default_tree_path)
hp.DATASET = "EU"
hp.LLM_API_BACKEND = "localModel"  # one of: openai, genai, vllm, localModel
hp.LLM = "google/gemma-2-2b-it" #gpt-4.1"
hp.LLM_API_TIMEOUT = 120
hp.LLM_API_MAX_RETRIES = 4
hp.LLM_MAX_CONCURRENT_CALLS = 16
hp.LLM_API_STAGGERING_DELAY = 0.05
hp.VLLM_BASE_URL = "http://localhost:8000/v1"
hp.REASONING_IN_TRAVERSAL_PROMPT = -1
hp.SUBSET = "fiqa"  # chooses the traversal relevance definition
hp.MAX_BEAM_SIZE = 8
hp.SEARCH_WITH_PATH_RELEVANCE = True
hp.NUM_LEAF_CALIB = 0
hp.RELEVANCE_CHAIN_FACTOR = 0.5
hp.MAX_PROMPT_PROTO_SIZE = None
hp.MAX_DOC_DESC_CHAR_LEN = None

LOG_DIR = REPO_ROOT / "results" / "tree_construction"
LOG_DIR.mkdir(parents=True, exist_ok=True)
logger = setup_logger("eu_conventions_traversal_notebook", str(LOG_DIR / "eu_conventions_traversal_notebook.log"))

hp


In [ ]:
tree_path = Path(hp.TREE_PATH)
tree_obj = pickle.loads(tree_path.read_bytes())
semantic_root_node = SemanticNode().load_dict(tree_obj) if isinstance(tree_obj, dict) else tree_obj
node_registry = compute_node_registry(semantic_root_node)
all_leaf_nodes = get_all_leaf_nodes_with_path(semantic_root_node)

print(f"Loaded tree from: {tree_path}")
print(f"Total nodes in semantic tree: {len(node_registry)}")
print(f"Total leaf nodes: {len(all_leaf_nodes)}")
print("Root description preview:")
print(semantic_root_node.desc[:800])


In [ ]:
def make_llm_api(hp, logger):
    if hp.LLM_API_BACKEND == "genai":
        llm_api = GenAIAPI(hp.LLM, logger=logger, timeout=hp.LLM_API_TIMEOUT, max_retries=hp.LLM_API_MAX_RETRIES)
    elif hp.LLM_API_BACKEND == "openai":
        llm_api = OpenAIResponsesAPI(hp.LLM, logger=logger, timeout=hp.LLM_API_TIMEOUT, max_retries=hp.LLM_API_MAX_RETRIES)
    elif hp.LLM_API_BACKEND == "vllm":
        llm_api = VllmAPI(hp.LLM, logger=logger, timeout=hp.LLM_API_TIMEOUT, max_retries=hp.LLM_API_MAX_RETRIES, base_url=hp.VLLM_BASE_URL)
    elif hp.LLM_API_BACKEND == "localModel":
        llm_api = LocalModelAPI(hp.LLM, logger=logger, timeout=hp.LLM_API_TIMEOUT, max_retries=hp.LLM_API_MAX_RETRIES, log_api_calls=True)
    else:
        raise ValueError(f"Unknown backend: {hp.LLM_API_BACKEND}")

    llm_api_kwargs = {
        "max_concurrent_calls": hp.LLM_MAX_CONCURRENT_CALLS,
        "response_mime_type": "application/json",
        "response_schema": get_traversal_prompt_response_constraint(bool(hp.REASONING_IN_TRAVERSAL_PROMPT)),
        "staggering_delay": hp.LLM_API_STAGGERING_DELAY,
        "print_summary_report": False,
        "thinking_config": genai_types.ThinkingConfig(thinking_budget=hp.REASONING_IN_TRAVERSAL_PROMPT),
    }

    if hp.LLM_API_BACKEND == "vllm":
        llm_api_kwargs.pop("response_mime_type")
        llm_api_kwargs.pop("thinking_config")
        llm_api_kwargs.pop("response_schema")

    if hp.LLM_API_BACKEND == "openai":
        llm_api_kwargs.pop("response_mime_type")
        llm_api_kwargs.pop("thinking_config")

    return llm_api, llm_api_kwargs


llm_api, llm_api_kwargs = make_llm_api(hp, logger)
print(type(llm_api).__name__)


## Example Queries

Pick one of these or write your own EU Convention query in the next cell.


In [ ]:
EXAMPLE_QUERIES = [
    "What does Article 6 say about the right to a fair trial?",
    "When can the right to liberty and security be restricted?",
    "What protections does Article 8 give for private and family life?",
    "What does the Convention say about prohibition of torture?",
    "How does Protocol 1 protect property rights?",
    "What are the rules for individual applications to the European Court of Human Rights?",
]

for idx, query in enumerate(EXAMPLE_QUERIES):
    print(f"[{idx}] {query}")


In [ ]:
query = EXAMPLE_QUERIES[0]
num_iters = 12
query


In [ ]:
def make_sample(query: str) -> InferSample:
    return InferSample(
        semantic_root_node,
        node_registry,
        hp=hp,
        logger=logger,
        query=query,
        gold_paths=[],
        excluded_ids_set=set(),
    )


async def run_one_iteration_async(sample: InferSample):
    inputs = sample.get_step_prompts()
    prompts = [prompt for prompt, _ in inputs]
    slates = [slate for _, slate in inputs]

    raw_responses = await llm_api.run_batch(prompts, **llm_api_kwargs)
    response_jsons = [post_process(output, return_json=True) for output in raw_responses]
    sample.update(slates, response_jsons)
    return raw_responses, response_jsons


def run_one_iteration(sample: InferSample):
    return run_coro_sync(run_one_iteration_async(sample))


def print_frontier(sample: InferSample):
    print("Current beam state paths:")
    for state_path in sample.beam_state_paths:
        node = state_path[-1]
        print(f"  path={node.path} | path_relevance={node.path_relevance:.3f} | desc={node.desc[:180]}")


def print_top_predictions(sample: InferSample, k: int = 5):
    top_preds = sample.get_top_predictions(k=k)
    if not top_preds:
        print("No leaf predictions yet. Increase num_iters.")
        return

    for rank, (node, score) in enumerate(top_preds, start=1):
        print(f"[{rank}] path={node.path} | score={score:.3f} | id={node.id}")
        print(node.desc[:600])
        print("-" * 120)


async def run_query_async(query: str, num_iters: int = 4, detailed_logs: bool = False) -> InferSample:
    sample = make_sample(query)
    if detailed_logs: print(f"Running traversal for query: {query}")
    for step in range(num_iters):
        print(f"\n--- Iteration {step + 1} ---")
        _, response_jsons = await run_one_iteration_async(sample)
        if(detailed_logs): print_frontier(sample)
        if response_jsons:
            first_reasoning = response_jsons[0].get("reasoning", "")
            if(detailed_logs):
                print("\nReasoning preview:")
                print(str(first_reasoning)[:800])
    return sample


def run_query(query: str, num_iters: int = 4, detailed_logs: bool = False) -> InferSample:
    return run_coro_sync(run_query_async(query, num_iters=num_iters, detailed_logs=detailed_logs))


In [ ]:
#sample = run_query(query, num_iters=num_iters)


In [ ]:
#print_top_predictions(sample, k=5)


## ECtHR Multi-Case Article Retrieval Test

This section loads ECtHR cases, joins each case's fact paragraphs into one LATTICE query, retrieves Convention/Protocol article leaves from the EU conventions tree, and compares the predicted article set with the dataset's multi-label article targets.

The main count is `all_gold_found`: a case is counted as correct when every expected article label appears somewhere in the top retrieved article set. Because a case may involve multiple articles, the table also reports partial matches, precision, recall, and F1.


In [ ]:
import re

import pandas as pd
from datasets import load_dataset


def load_ecthr_dataset(split: str = "train", config: str = "alleged-violation-prediction"):
    """Load ECtHR Cases directly from Hugging Face.

    `alleged-violation-prediction` is the task that matches this experiment:
    given the case facts, predict the Convention articles that may be related
    to / alleged in the case. Use `violation-prediction` if you instead want
    the articles the Court found violated.

    The dataset uses a loading script, so `trust_remote_code=True` is required.
    """
    dataset = load_dataset(
        "AUEB-NLP/ecthr_cases",
        config,
        split=split,
        trust_remote_code=True,
    )
    return dataset


def get_label_names(dataset, label_column: str = "labels") -> list[str] | None:
    """No label-name lookup is needed for `AUEB-NLP/ecthr_cases`.

    The dataset already stores labels as article strings, e.g. `"6"` or
    `"P1-1"`. The argument is kept so the rest of the notebook stays generic.
    """
    return None


def normalize_article_label(value) -> str | None:
    """Normalize labels from ECtHR Cases or article text.

    Examples:
    - 6 -> article_6
    - "Article 6" -> article_6
    - "P1-1" -> protocol_1_article_1
    - "article_1_of_protocol_1" -> protocol_1_article_1
    """
    if value is None:
        return None
    s = str(value).strip().lower()
    if not s:
        return None

    s = s.replace("no violation", "")
    s = s.replace("non violation", "")
    s = s.replace("non-violation", "")
    s = s.replace(".", "")
    s = s.replace("-", "_")
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"^echr_", "", s)
    s = re.sub(r"^convention_", "", s)

    # ECtHR Cases labels bare Convention articles as strings: "2", "3", "6"...
    if re.fullmatch(r"\d+[a-z]?", s):
        return f"article_{s}"

    # Compact protocol labels appear as P1-1, P4-2, etc. after normalization: p1_1.
    m = re.fullmatch(r"p(\d+)_(\d+)", s)
    if m:
        return f"protocol_{int(m.group(1))}_article_{int(m.group(2))}"

    # article_1_of_protocol_1 / article_1_protocol_1
    m = re.search(r"article_?(\d+[a-z]?)_(?:of_)?protocol_?(\d+)", s)
    if m:
        return f"protocol_{int(m.group(2))}_article_{m.group(1)}"

    # protocol_1_article_1
    m = re.search(r"protocol_?(\d+)_article_?(\d+[a-z]?)", s)
    if m:
        return f"protocol_{int(m.group(1))}_article_{m.group(2)}"

    m = re.search(r"article_?(\d+[a-z]?)", s)
    if m:
        return f"article_{m.group(1)}"

    return None


def example_gold_articles(example: dict, label_names: list[str] | None = None, label_column: str = "labels") -> set[str]:
    """Convert one ECtHR example's labels to normalized article IDs."""
    raw_labels = example.get(label_column, [])
    if raw_labels is None:
        return set()
    if isinstance(raw_labels, (int, str)):
        raw_labels = [raw_labels]

    gold = set()
    for raw_label in raw_labels:
        label_value = label_names[raw_label] if label_names and isinstance(raw_label, int) else raw_label
        normalized = normalize_article_label(label_value)
        if normalized:
            gold.add(normalized)
    return gold



In [ ]:
TREE_ARTICLE_RE = re.compile(
    r"(?:Protocol\s+(?P<protocol>\d+)\s+)?Art\.\s*(?P<article>\d+[A-Za-z]?)",
    flags=re.IGNORECASE,
)


def facts_to_case_prompt(facts: list[str], max_chars: int = 12000) -> str:
    """Join ECtHR fact paragraphs into one retrieval query for LATTICE."""
    fact_text = "\n".join(f"- {fact.strip()}" for fact in facts if str(fact).strip())
    if len(fact_text) > max_chars:
        fact_text = fact_text[:max_chars] + "\n- [facts truncated]"

    return f"""You are an LLM traversing a semantic knowledge tree of European Convention law.

The tree is structured as follows:
- Internal nodes are semantic summaries of legal concepts or clusters of rights.
- Each internal node summarizes the information contained in its child nodes.
- Leaf nodes are specific Convention or Protocol articles and their provisions.

Given the facts of an ECtHR case, traverse the tree from the root toward the most legally relevant leaf articles.

Your task is not to retrieve every article that is semantically related. Your task is to identify the articles most likely to be used in an application before the European Court of Human Rights.

Traversal rules:
1. First identify the central legal injury in the facts.
2. Move only into child nodes whose summary directly matches that injury.
3. Penalize branches that are only factually mentioned, background-related, or speculative.
4. Prefer branches where the facts show an actual state interference, omission, procedural defect, or lack of remedy.
5. Continue traversal until you reach the strongest article leaves.
6. Return only leaf articles whose full path from the root is strongly supported by the facts.

For each selected article leaf:
- Give the article number and title.
- Explain the semantic path that led to it.
- Explain why it is a primary or secondary article.
- Retrieve the relevant provision text.
- Assign a relevance score from 0 to 1.

Do not include weak articles unless they are necessary to explain why they were rejected.

Case facts:
{fact_text}
"""


def extract_articles_from_tree_text(text: str) -> set[str]:
    """Extract normalized article IDs from retrieved EU conventions tree leaves."""
    predictions = set()
    for match in TREE_ARTICLE_RE.finditer(text or ""):
        article = match.group("article").lower()
        protocol = match.group("protocol")
        if protocol:
            predictions.add(f"protocol_{int(protocol)}_article_{article}")
        else:
            predictions.add(f"article_{article}")
    return predictions


def predicted_articles_from_sample(sample: InferSample, k: int = 10, min_score: float | None = None, max_articles: int | None = None) -> tuple[set[str], list[dict]]:
    """Take top LATTICE leaf predictions and extract article IDs from their text.

    Lower k, higher min_score, or lower max_articles generally improves precision
    at the cost of recall.
    """
    predicted = set()
    rows = []
    for rank, (node, score) in enumerate(sample.get_top_predictions(k=k), start=1):
        article_ids = extract_articles_from_tree_text(node.desc)
        passes_filters = bool(article_ids) and (min_score is None or float(score) >= min_score)
        included = False
        if passes_filters:
            for article_id in sorted(article_ids):
                if max_articles is not None and len(predicted) >= max_articles:
                    break
                predicted.add(article_id)
                included = True
        rows.append(
            {
                "rank": rank,
                "score": float(score),
                "node_id": node.id,
                "included": included,
                "articles": sorted(article_ids),
                "text": node.desc[:800],
            }
        )
    return predicted, rows


ARTICLE_SELECTOR_RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "reasoning": {
            "type": "string",
            "description": "Brief explanation for selecting or rejecting the candidate articles.",
        },
        "selected_articles": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Normalized article IDs selected from the provided candidate list only.",
        },
    },
    "required": ["reasoning", "selected_articles"],
}


def article_id_to_display(article_id: str) -> str:
    m = re.fullmatch(r"article_(\d+[a-z]?)", article_id)
    if m:
        return f"Article {m.group(1).upper()}"
    m = re.fullmatch(r"protocol_(\d+)_article_(\d+[a-z]?)", article_id)
    if m:
        return f"Protocol {m.group(1)}, Article {m.group(2).upper()}"
    return article_id


def case_facts_from_query(query: str) -> str:
    marker = "Case facts:\n"
    return query.split(marker, 1)[-1].strip() if marker in query else query.strip()


def aggregate_candidate_articles(top_rows: list[dict], max_evidence_per_article: int = 2) -> list[dict]:
    candidates_by_id = {}
    for row in top_rows:
        for article_id in row.get("articles", []):
            candidate = candidates_by_id.setdefault(
                article_id,
                {
                    "article_id": article_id,
                    "label": article_id_to_display(article_id),
                    "best_rank": row["rank"],
                    "best_score": row["score"],
                    "evidence": [],
                },
            )
            if row["rank"] < candidate["best_rank"]:
                candidate["best_rank"] = row["rank"]
            if row["score"] > candidate["best_score"]:
                candidate["best_score"] = row["score"]
            if len(candidate["evidence"]) < max_evidence_per_article:
                candidate["evidence"].append(row["text"])
    return sorted(candidates_by_id.values(), key=lambda item: (item["best_rank"], -item["best_score"], item["article_id"]))


def build_article_selector_prompt(query: str, candidate_articles: list[dict], max_articles: int | None = None) -> str:
    limit_instruction = (
        f"Select at most {max_articles} article IDs. Prefer fewer articles when uncertain."
        if max_articles is not None
        else "Select all and only the candidate article IDs that are strongly supported. Prefer fewer articles when uncertain."
    )
    candidate_text = []
    for candidate in candidate_articles:
        evidence = "\n".join(f"    Evidence {idx + 1}: {text}" for idx, text in enumerate(candidate["evidence"]))
        candidate_text.append(
            f"- {candidate['article_id']} ({candidate['label']}) | "
            f"best_rank={candidate['best_rank']} | best_score={candidate['best_score']:.3f}\n{evidence}"
        )

    return f"""You are doing the final ECtHR alleged-violation article selection after a LATTICE retrieval step.

Choose only articles that are likely to be alleged violations for this case. Do not choose articles that are merely background, procedural context, or weakly related. {limit_instruction}

Return JSON with:
- reasoning: a brief explanation
- selected_articles: normalized IDs copied exactly from the candidate list

Case facts:
{case_facts_from_query(query)}

Candidate articles from LATTICE:
{chr(10).join(candidate_text)}
"""


def parse_selector_response(output: str, allowed_articles: set[str]) -> dict:
    try:
        parsed = post_process(output, return_json=True)
    except Exception as exc:
        parsed = {"reasoning": f"Selector response could not be parsed: {exc}", "selected_articles": []}
    selected = []
    for article in parsed.get("selected_articles", []):
        normalized = normalize_article_label(article)
        if normalized in allowed_articles and normalized not in selected:
            selected.append(normalized)
    return {"reasoning": parsed.get("reasoning", ""), "selected_articles": selected}


async def select_articles_with_llm_batch_async(queries: list[str], top_rows_list: list[list[dict]], max_articles: int | None = None) -> list[dict]:
    candidate_lists = [aggregate_candidate_articles(top_rows) for top_rows in top_rows_list]
    prompts = []
    prompt_indexes = []
    results = []
    for idx, (query, candidate_articles) in enumerate(zip(queries, candidate_lists)):
        allowed_articles = {candidate["article_id"] for candidate in candidate_articles}
        results.append(
            {
                "reasoning": "No candidate articles were available for selector.",
                "selected_articles": [],
                "candidate_articles": sorted(allowed_articles),
            }
        )
        if candidate_articles:
            prompts.append(build_article_selector_prompt(query, candidate_articles, max_articles=max_articles))
            prompt_indexes.append(idx)

    if not prompts:
        return results

    selector_kwargs = {
        **llm_api_kwargs,
        "response_schema": ARTICLE_SELECTOR_RESPONSE_SCHEMA,
        "print_summary_report": False,
    }
    raw_outputs = await llm_api.run_batch(prompts, **selector_kwargs)
    for result_idx, output in zip(prompt_indexes, raw_outputs):
        allowed_articles = set(results[result_idx]["candidate_articles"])
        parsed = parse_selector_response(output, allowed_articles)
        parsed["candidate_articles"] = results[result_idx]["candidate_articles"]
        results[result_idx] = parsed
    return results


def score_prediction(gold: set[str], predicted: set[str]) -> dict:
    """Score one multi-label case."""
    true_positive = len(gold & predicted)
    precision = true_positive / len(predicted) if predicted else 0.0
    recall = true_positive / len(gold) if gold else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if precision + recall else 0.0
    return {
        "any_gold_found": bool(gold & predicted),
        "all_gold_found": bool(gold) and gold.issubset(predicted),
        "exact_set_match": bool(gold) and gold == predicted,
        "true_positive": true_positive,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def run_ecthr_case(example: dict, label_names: list[str] | None, *, num_iters: int = 6, top_k: int = 10, prediction_min_score: float | None = None, max_predicted_articles: int | None = None, use_llm_selector: bool = False):
    facts = example.get("text") or example.get("facts") or []
    if isinstance(facts, str):
        facts = [facts]

    query = facts_to_case_prompt(facts)
    sample = run_query(query, num_iters=num_iters)
    predicted, top_rows = predicted_articles_from_sample(sample, k=top_k, min_score=prediction_min_score, max_articles=max_predicted_articles)
    lattice_predicted = set(predicted)
    selector_result = None
    
    if use_llm_selector:
        selector_result = run_coro_sync(select_articles_with_llm_batch_async([query], [top_rows], max_articles=max_predicted_articles))[0]
        predicted = set(selector_result["selected_articles"])
        
        
    gold = example_gold_articles(example, label_names)
    gold_removed_by_selector_articles = sorted((gold & lattice_predicted) - predicted) if selector_result else []
    metrics = score_prediction(gold, predicted)

    return {
        "query": query,
        "gold": sorted(gold),
        "predicted": sorted(predicted),
        "lattice_predicted": sorted(lattice_predicted),
        "gold_removed_by_selector": len(gold_removed_by_selector_articles),
        "gold_removed_by_selector_articles": gold_removed_by_selector_articles,
        "top_rows": top_rows,
        "selector_result": selector_result,
        "sample": sample,
        **metrics,
    }
    


async def run_lattice_iterations_for_samples_async(samples: list[InferSample], *, num_iters: int, detailed_logs: bool = False):
    """Run traversal iterations for many samples using one shared async LLM batch per step."""
    for step in range(num_iters):
        print(f"\n--- Batched iteration {step + 1}/{num_iters} ({len(samples)} cases) ---")
        inputs_by_sample = [sample.get_step_prompts() for sample in samples]
        counts = [len(inputs) for inputs in inputs_by_sample]
        flat_inputs = [item for inputs in inputs_by_sample for item in inputs]
        if not flat_inputs:
            print("No prompts left to process.")
            break

        flat_prompts = [prompt for prompt, _ in flat_inputs]
        flat_slates = [slate for _, slate in flat_inputs]
        raw_responses = await llm_api.run_batch(flat_prompts, **llm_api_kwargs)
        flat_response_jsons = [post_process(output, return_json=True) for output in raw_responses]

        offset = 0
        for sample, count in zip(samples, counts):
            sample_slates = flat_slates[offset:offset + count]
            sample_response_jsons = flat_response_jsons[offset:offset + count]
            if count:
                sample.update(sample_slates, sample_response_jsons)
                if detailed_logs:
                    print_frontier(sample)
            offset += count


async def evaluate_ecthr_cases_async(dataset, label_names: list[str] | None, *, n_cases: int = 10, num_iters: int = 6, top_k: int = 10, start: int = 0, detailed_logs: bool = False, prediction_min_score: float | None = None, max_predicted_articles: int | None = None, use_llm_selector: bool = False):
    """Evaluate many ECtHR cases by batching all LATTICE prompts across cases at each iteration."""
    selected = list(dataset.select(range(start, min(start + n_cases, len(dataset)))))
    samples = []
    queries = []
    for example in selected:
        facts = example.get("text") or example.get("facts") or []
        if isinstance(facts, str):
            facts = [facts]
        query = facts_to_case_prompt(facts)
        queries.append(query)
        samples.append(make_sample(query))

    await run_lattice_iterations_for_samples_async(samples, num_iters=num_iters, detailed_logs=detailed_logs)

    lattice_predictions = []
    top_rows_list = []
    for sample in samples:
        predicted, top_rows = predicted_articles_from_sample(sample, k=top_k, min_score=prediction_min_score, max_articles=max_predicted_articles)
        lattice_predictions.append(predicted)
        top_rows_list.append(top_rows)

    selector_results = [None] * len(samples)
    
    if use_llm_selector:
        selector_results = await select_articles_with_llm_batch_async(queries, top_rows_list, max_articles=max_predicted_articles)

    results = []
    for local_idx, (example, query, sample, lattice_predicted, top_rows, selector_result) in enumerate(zip(selected, queries, samples, lattice_predictions, top_rows_list, selector_results)):
        case_idx = start + local_idx
        predicted = set(selector_result["selected_articles"]) if selector_result else lattice_predicted
        gold = example_gold_articles(example, label_names)
        gold_removed_by_selector_articles = sorted((gold & lattice_predicted) - predicted) if selector_result else []
        metrics = score_prediction(gold, predicted)
        result = {
            "case_index": case_idx,
            "query": query,
            "gold": sorted(gold),
            "predicted": sorted(predicted),
            "lattice_predicted": sorted(lattice_predicted),
            "gold_removed_by_selector": len(gold_removed_by_selector_articles),
            "gold_removed_by_selector_articles": gold_removed_by_selector_articles,
            "top_rows": top_rows,
            "selector_result": selector_result,
            "sample": sample,
            **metrics,
        }
        results.append(result)
        print(f"\n================ ECtHR case {case_idx} ================")
        print("Gold:     ", result["gold"])
        print("Predicted: ", result["predicted"])
        print(
            f"Correct(all gold found) : {result['all_gold_found']} | "
            f"Recall: {result['recall']:.2f} | Precision: {result['precision']:.2f} | F1: {result['f1']:.2f}"
        )
        if selector_result:
            print(f"Gold articles removed by selector: {result['gold_removed_by_selector']} {result['gold_removed_by_selector_articles']}")

    df = pd.DataFrame(
        [
            {
                "case_index": r["case_index"],
                "gold": r["gold"],
                "predicted": r["predicted"],
                "lattice_predicted": r.get("lattice_predicted", r["predicted"]),
                "gold_removed_by_selector": r.get("gold_removed_by_selector", 0),
                "gold_removed_by_selector_articles": r.get("gold_removed_by_selector_articles", []),
                "any_gold_found": r["any_gold_found"],
                "all_gold_found": r["all_gold_found"],
                "exact_set_match": r["exact_set_match"],
                "true_positive": r["true_positive"],
                "precision": r["precision"],
                "recall": r["recall"],
                "f1": r["f1"],
            }
            for r in results
        ]
    )
    print("\n================ Summary ================")
    print_final_average_ecthr_cases_batch(df)
    return df, results


def evaluate_ecthr_cases_batched(dataset, label_names: list[str] | None, *, n_cases: int = 10, num_iters: int = 6, top_k: int = 10, start: int = 0, detailed_logs: bool = False, prediction_min_score: float | None = None, max_predicted_articles: int | None = None, use_llm_selector: bool = False):
    return run_coro_sync(
        evaluate_ecthr_cases_async(
            dataset,
            label_names,
            n_cases=n_cases,
            num_iters=num_iters,
            top_k=top_k,
            start=start,
            detailed_logs=detailed_logs,
            prediction_min_score=prediction_min_score,
            max_predicted_articles=max_predicted_articles,
            use_llm_selector=use_llm_selector,
        )
    )


def print_final_average_ecthr_cases_batch(df):
    summary = pd.DataFrame(
        [
            {
                "cases_evaluated": len(df),
                "any_gold_found": int(df["any_gold_found"].sum()) / len(df),
                "all_gold_found": int(df["all_gold_found"].sum()) / len(df),
                "exact_set_match": int(df["exact_set_match"].sum()) / len(df),
                "mean_gold_removed_by_selector": df["gold_removed_by_selector"].mean() if "gold_removed_by_selector" in df else 0.0,
                "mean_recall": df["recall"].mean(),
                "mean_precision": df["precision"].mean(),
                "mean_f1": df["f1"].mean(),
            }
        ]
    )
    
    display(summary)
    return summary



def evaluate_ecthr_cases(dataset, label_names: list[str] | None, *, n_cases: int = 10, num_iters: int = 6, top_k: int = 10, start: int = 0, prediction_min_score: float | None = None, max_predicted_articles: int | None = None, use_llm_selector: bool = False):
    """Loop over multiple ECtHR cases and count how often LATTICE finds the expected articles."""
    results = []
    for local_idx, example in enumerate(dataset.select(range(start, min(start + n_cases, len(dataset))))):
        case_idx = start + local_idx
        print(f"\n================ ECtHR case {case_idx} ================")
        result = run_ecthr_case(example, label_names, num_iters=num_iters, top_k=top_k, prediction_min_score=prediction_min_score, max_predicted_articles=max_predicted_articles, use_llm_selector=use_llm_selector)
        result["case_index"] = case_idx
        results.append(result)
        print("Gold:     ", result["gold"])
        print("Predicted: ", result["predicted"])
        print(
            f"Correct(all gold found) : {result['all_gold_found']} | "
            f"Recall: {result['recall']:.2f} | Precision: {result['precision']:.2f} | F1: {result['f1']:.2f}"
        )
        if result.get("selector_result"):
            print(f"Gold articles removed by selector: {result['gold_removed_by_selector']} {result['gold_removed_by_selector_articles']}")
    

    df = pd.DataFrame(
        [
            {
                "case_index": r["case_index"],
                "gold": r["gold"],
                "predicted": r["predicted"],
                "lattice_predicted": r.get("lattice_predicted", r["predicted"]),
                "gold_removed_by_selector": r.get("gold_removed_by_selector", 0),
                "gold_removed_by_selector_articles": r.get("gold_removed_by_selector_articles", []),
                "any_gold_found": r["any_gold_found"],
                "all_gold_found": r["all_gold_found"],
                "exact_set_match": r["exact_set_match"],
                "true_positive": r["true_positive"],
                "precision": r["precision"],
                "recall": r["recall"],
                "f1": r["f1"],
            }
            for r in results
        ]
    )
    print("\n================ Summary ================")
    """print(f"Cases evaluated: {len(df)}")
    print(f"Any gold article found: {int(df['any_gold_found'].sum())}/{len(df)}")
    print(f"All gold articles found: {int(df['all_gold_found'].sum())}/{len(df)}")
    print(f"Exact set match: {int(df['exact_set_match'].sum())}/{len(df)}")
    print(f"Mean recall: {df['recall'].mean():.3f}")
    print(f"Mean precision: {df['precision'].mean():.3f}")
    print(f"Mean F1: {df['f1'].mean():.3f}")"""
    print_final_average_ecthr_cases_batch(df)
    return df, results



In [ ]:
# Configure the ECtHR evaluation run.
# Use split="train" if you want to test quickly on the training set facts,
# or split="test" for the held-out ECtHR Cases test split.
ECTHR_CONFIG = "alleged-violation-prediction"
ECTHR_SPLIT = "train"
N_CASES = 1
NUM_ITERS_PER_CASE = 9
TOP_K_LEAVES = 10
PREDICTION_MIN_SCORE = 0.4  # Try 0.25 or 0.35 for stricter precision filtering.
MAX_PREDICTED_ARTICLES = None  # Lower values usually improve precision and reduce recall.
USE_LLM_SELECTOR = True  # Adds one final LLM call per case to select from LATTICE candidates.

ecthr_dataset = load_ecthr_dataset(split=ECTHR_SPLIT, config=ECTHR_CONFIG)
ecthr_label_names = get_label_names(ecthr_dataset)

print(ecthr_dataset)
print("Label names:", ecthr_label_names)
print("First example labels:", ecthr_dataset[0]["labels"])

results_df, ecthr_results = evaluate_ecthr_cases_batched(
    ecthr_dataset,
    ecthr_label_names,
    n_cases=N_CASES,
    num_iters=NUM_ITERS_PER_CASE,
    top_k=TOP_K_LEAVES,
    prediction_min_score=PREDICTION_MIN_SCORE,
    max_predicted_articles=MAX_PREDICTED_ARTICLES,
    use_llm_selector=USE_LLM_SELECTOR,
)

results_df


In [ ]:
print_final_average_ecthr_cases_batch(results_df)

In [ ]:
ecthr_dataset[0]

In [ ]:
ecthr_results

In [ ]:
# Optional: visualize the prediction tree explored by the LLM.
VIS_DIR = REPO_ROOT / "results" / "tree_construction"
VIS_DIR.mkdir(parents=True, exist_ok=True)
html_path = VIS_DIR / "eu_conventions_traversal_prediction_tree.html"

fig = visualize_sample(sample, width=1400, height=900, save_path=str(html_path))
print(html_path)


## Notes

- `num_iters` controls how deep the traversal goes before you inspect results.
- If `print_top_predictions(...)` says there are no leaf predictions yet, increase `num_iters`.
- `hp.SUBSET` controls the relevance definition used inside the traversal prompt. For legal QA-style usage, `fiqa` is usually a reasonable default here.
- `visualize_sample(...)` shows the explored prediction tree, not the static semantic tree structure.
- The default tree path first checks `trees/EU/eu_conventions_notebook/tree-bottom-up-llm.pkl`, then falls back to the existing `trees/EU/codigo_civil_notebook/tree-bottom-up-llm.pkl` path.

- The ECtHR section uses `AUEB-NLP/ecthr_cases` (`alleged-violation-prediction` by default). The case facts are joined into one query and the label list is treated as the expected multi-article target set.
